# Module 6 Homework — Batch Processing with Spark

Yellow Taxi Trip Data, November 2025

## Download data

In [1]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
!wget -q https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
!ls -lh yellow_tripdata_2025-11.parquet taxi_zone_lookup.csv

-rw-r--r-- 1 jovyan users 13K Feb 22  2024 taxi_zone_lookup.csv
-rw-r--r-- 1 jovyan users 68M Dec 19 15:51 yellow_tripdata_2025-11.parquet


## Question 1: Install Spark and PySpark

Create a local Spark session and check `spark.version`.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework_06") \
    .getOrCreate()

spark.version

'3.5.0'

## Question 2: Yellow November 2025

Read the parquet file, repartition to 4 partitions, save and check average file size.

In [3]:
df_yellow = spark.read.parquet("yellow_tripdata_2025-11.parquet")

df_yellow.repartition(4).write.parquet("yellow_2025_11_repartitioned", mode="overwrite")

In [4]:
import os, glob

parquet_files = glob.glob("yellow_2025_11_repartitioned/*.parquet")
sizes = [os.path.getsize(f) for f in parquet_files]
avg_mb = sum(sizes) / len(sizes) / (1024 * 1024)

print(f"Number of parquet files: {len(parquet_files)}")
print(f"Average file size: {avg_mb:.1f} MB")

Number of parquet files: 4
Average file size: 24.4 MB


## Register temp views for SQL queries

In [5]:
df_yellow.createOrReplaceTempView("yellow_trips")

df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
df_zones.createOrReplaceTempView("zones")

## Question 3: Count records

How many taxi trips were there on the 15th of November?

In [6]:
spark.sql("""
    SELECT COUNT(*) AS trip_count
    FROM yellow_trips
    WHERE CAST(tpep_pickup_datetime AS DATE) = '2025-11-15'
""").show()

+----------+
|trip_count|
+----------+
|    162604|
+----------+



## Question 4: Longest trip

What is the length of the longest trip in hours?

In [7]:
spark.sql("""
    SELECT
        ROUND(
            MAX(
                (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600
            ), 1
        ) AS longest_trip_hours
    FROM yellow_trips
""").show()

+------------------+
|longest_trip_hours|
+------------------+
|              90.6|
+------------------+



## Question 5: User Interface

Spark's User Interface runs on port **4040**.

## Question 6: Least frequent pickup location zone

Join yellow trips with zone lookup to find the least frequent pickup location.

In [9]:
spark.sql("""
    SELECT
        z.Zone,
        COUNT(*) AS pickup_count
    FROM yellow_trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY pickup_count ASC
    LIMIT 10
""").show(truncate=False)

+---------------------------------------------+------------+
|Zone                                         |pickup_count|
+---------------------------------------------+------------+
|Governor's Island/Ellis Island/Liberty Island|1           |
|Eltingville/Annadale/Prince's Bay            |1           |
|Arden Heights                                |1           |
|Port Richmond                                |3           |
|Rikers Island                                |4           |
|Rossville/Woodrow                            |4           |
|Great Kills                                  |4           |
|Green-Wood Cemetery                          |4           |
|Jamaica Bay                                  |5           |
|Westerleigh                                  |12          |
+---------------------------------------------+------------+



In [ ]:
spark.stop()